In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import random
import math
import copy
import cv2

from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, Callback
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.xception import preprocess_input, decode_predictions
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing import image_dataset_from_directory
#from tensorflow.keras.layers.experimental.preprocessing import RandomFlip, RandomRotation
from tensorflow.keras.layers import RandomFlip, RandomRotation

from keras import backend as K
from sklearn.metrics import classification_report, confusion_matrix

2025-07-13 22:15:17.847282: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752444918.077808      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752444918.151330      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
! conda install -y gdown

/bin/bash: line 1: conda: command not found


In [3]:
import gdown

# a file
url = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/bwh3zbpkpv-1.zip"
gdown.download(url)

Downloading...
From: https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/bwh3zbpkpv-1.zip
To: /kaggle/working/bwh3zbpkpv-1.zip
100%|██████████| 8.44G/8.44G [05:58<00:00, 23.6MB/s]  


'bwh3zbpkpv-1.zip'

In [4]:
import zipfile
import os

zip_path = "/kaggle/working/bwh3zbpkpv-1.zip"
extract_path = "/kaggle/working/bwh3zbpkpv"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped to:", extract_path)


Unzipped to: /kaggle/working/bwh3zbpkpv


In [5]:
# List the top-level files and folders
for root, dirs, files in os.walk(extract_path):
    print("Current folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files)
    break  # Remove this 'break' to see everything recursively


Current folder: /kaggle/working/bwh3zbpkpv
Subfolders: ['Dataset for Crop Pest and Disease Detection']
Files: []


In [6]:
import os

inner_folder = os.path.join(extract_path, "Dataset for Crop Pest and Disease Detection")

for root, dirs, files in os.walk(inner_folder):
    print("Current folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files)
    break  # remove break if you want to go deeper


Current folder: /kaggle/working/bwh3zbpkpv/Dataset for Crop Pest and Disease Detection
Subfolders: ['Raw Data', 'CCMT Dataset-Augmented']
Files: []


In [7]:
ccmt_path = os.path.join(inner_folder, "CCMT Dataset-Augmented")

for root, dirs, files in os.walk(ccmt_path):
    print("Current folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files)
    break


Current folder: /kaggle/working/bwh3zbpkpv/Dataset for Crop Pest and Disease Detection/CCMT Dataset-Augmented
Subfolders: ['Cashew', 'Maize', 'Cassava', 'Tomato']
Files: []


In [8]:
train_dir = os.path.join(
    inner_folder,
    "CCMT Dataset-Augmented",
    "Cashew",
    "train_set"
)

test_dir = os.path.join(
    inner_folder,
    "CCMT Dataset-Augmented",
    "Cashew",
    "test_set"
)

# validation_dir = os.path.join(
#     inner_folder,
#     "Raw Data",
#     "CCMT Dataset",
#     "Cashew",
# )
validation_dir = os.path.join(
    inner_folder,
    "CCMT Dataset-Augmented",
    "Cashew",
    "test_set"
)

In [9]:
# from google.colab import drive
# drive.mount('/content/drive')

# define train forms

In [10]:
# train_dir = "/content/drive/My Drive/Dataset/Cashew_train/train_set"
# test_dir = "/content/drive/My Drive/Dataset/Cashew_train/test_set"
# validation_dir = "/content/drive/My Drive/Dataset/Cashew"

In [11]:
import os

train_data = []

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):  # ✅ only process directories
        for file_name in os.listdir(class_path):
            file_path = os.path.join(class_path, file_name)
            train_data.append((file_path, class_name))

In [12]:
train_df = pd.DataFrame(train_data, columns=['File_Path', 'Class_Name'])
train_df.head()

,File_Path,Class_Name
0,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877
1,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877
2,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877
3,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877
4,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877


In [13]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
train_df['Class_ID'] = encoder.fit_transform(train_df['Class_Name'])
train_df.head()

,File_Path,Class_Name,Class_ID
0,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877,2
1,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877,2
2,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877,2
3,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877,2
4,/kaggle/working/bwh3zbpkpv/Dataset for Crop Pe...,healthy5877,2


In [14]:
train_df['Class_Name'].value_counts()

Class_Name
healthy5877        5877
red rust4751       4751
leaf miner3466     3466
anthracnose3102    3102
gumosis1714        1714
Name: count, dtype: int64

In [15]:
train_df['Class_ID'].value_counts()

Class_ID
2    5877
4    4751
3    3466
0    3102
1    1714
Name: count, dtype: int64

#take a look at the various diseases

In [16]:
#name is the title
def plotimage(desired_class: str,name):
    desired_class_df = train_df[train_df['Class_Name'] == desired_class]

    num_images_to_plot = 4

    fig, axes = plt.subplots(1, num_images_to_plot, figsize=(15, 5))

    for i, (index, row) in enumerate(desired_class_df.head(num_images_to_plot).iterrows()):
        image_path = row['File_Path']
        image = load_img(image_path)

        axes[i].imshow(image)
        axes[i].set_title(f"Image {i+1}: {name}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [17]:
#plotimage('healthy','HEALTHY')

In [18]:
#plotimage('red rust','RED RUST')

In [19]:
#plotimage('leaf miner','LEAF MINER')

In [20]:
#plotimage('anthracnose','ANTHRACNOSE')

In [21]:
#plotimage('gumosis','GUMOSIS')

# xception model

In [22]:
opt = tf.keras.optimizers.Adam(learning_rate=1e-4)
loss = 'categorical_crossentropy'
metrics = ['accuracy']

batch_size = 32

I0000 00:00:1752445354.760073      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [23]:
model1 = Xception(include_top=False,input_shape=(299, 299, 3), weights='imagenet')

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [24]:
input_shape= (299, 299)

In [25]:
datagen_train = ImageDataGenerator(rescale=1./255,
                                  width_shift_range=0.2,
                                  height_shift_range=0.2,
                                  zoom_range=0.2,
                                  horizontal_flip=True,
                                  vertical_flip=False)


datagen_test = ImageDataGenerator(rescale=1./255)
datagen_val = ImageDataGenerator(rescale=1./255)


generator_train = datagen_train.flow_from_directory(directory=train_dir,
                                                    target_size=input_shape,
                                                    batch_size=batch_size,
                                                    shuffle=True)

generator_test = datagen_test.flow_from_directory(directory=test_dir,
                                                  target_size=input_shape,
                                                  batch_size=batch_size,
                                                  shuffle=False)

generator_val = datagen_val.flow_from_directory(directory=validation_dir,
                                                  target_size=input_shape,
                                                  batch_size=batch_size,
                                                  shuffle=False)


Found 18910 images belonging to 5 classes.
Found 6901 images belonging to 5 classes.
Found 6901 images belonging to 5 classes.


In [26]:
math.ceil(generator_train.samples)

18910

In [27]:
next(generator_train)[1]

array([[0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0.]], dtype=float32)

# Xception deep network

In [28]:
conv_model = Model(inputs=model1.input, outputs=model1.output)

In [29]:
new_model = Sequential()
new_model.add(conv_model)
new_model.add(Flatten())
new_model.add(Dropout(0.5))
new_model.add(Dense(512, activation='relu'))
new_model.add(Dense(5, activation='softmax'))

In [30]:
new_model.compile(optimizer= opt, loss=loss, metrics=metrics)

In [31]:
num_iters = 30000
num_batches_train = generator_train.n // batch_size

epochs = int(num_iters / num_batches_train)
epochs = 10
print("Epoch: ",epochs)
desired_train_accuracy = 0.99

steps_per_epoch = generator_train.n // batch_size
steps_val = generator_val.n // batch_size
print("Steps_per_epoch: ",steps_per_epoch)
print("Steps_val: ",steps_val)

Epoch:  10
Steps_per_epoch:  590
Steps_val:  215


In [ ]:
Checkpoint = ModelCheckpoint("xception_cassava.keras", monitor="val_accuracy", save_best_only=True, mode="max")
EarlyStop = EarlyStopping(monitor="val_accuracy", baseline=desired_train_accuracy, patience=10, restore_best_weights=True, mode="auto")
EarlyStop = EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True,mode="auto")
history = new_model.fit(generator_train,
                        epochs=epochs,
                        callbacks=[Checkpoint, EarlyStop],
                        steps_per_epoch=steps_per_epoch,
                        validation_data=generator_val,
                        validation_steps=steps_val)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1752445396.236226      98 service.cc:148] XLA service 0x7e7828003e20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1752445396.236989      98 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1752445399.720600      98 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1752445415.819077      98 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1752445416.058321      98 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1752445417.414141      98 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1752445417.6570

142/590 ━━━━━━━━━━━━━━━━━━━━ 5:16 707ms/step - accuracy: 0.6454 - loss: 1.4312

In [ ]:
# save exception model

In [ ]:
import shutil

shutil.move("xception_cassava.keras", "/kaggle/working/xception_cassava.keras")

In [ ]:
import os

print(os.listdir('/kaggle/working'))


In [ ]:
new_model.save('/kaggle/working/my_model.h5')

In [ ]:
import os

print(os.listdir('/kaggle/working'))


In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/my_model.h5') 